In [1]:
import os
import sys
from pathlib import Path

# Проверяем, запускался ли уже этот блок
if "IS_ROOT_SET" not in globals():
    project_root = Path.cwd().parent.parent
    os.chdir(project_root)

    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

    # Создаем флаг-индикатор
    IS_ROOT_SET = True
    print("Корневая директория инициализирована.")
else:
    print("Инициализация уже была выполнена ранее.")


Корневая директория инициализирована.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
import lightgbm as lgb
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.calibration import CalibratedClassifierCV
import random

# Пол LGBMClassifier: https://habr.com/ru/companies/skillfactory/articles/530594/

In [3]:
data = pd.read_parquet("lab5/data/processed/processed_sepsis_data.parquet")

In [4]:
def prepare_target_and_split(df: pd.DataFrame, feature_cols: list) -> tuple:
    df = df.copy()

    gkf = GroupKFold(n_splits=5)
    train_idx, test_idx = next(gkf.split(df, groups=df["Patient_ID"]))

    train_df = df.iloc[train_idx]
    test_df = df.iloc[test_idx]

    X_train, y_train = train_df[feature_cols], train_df["SepsisLabel"]
    X_test, y_test = test_df[feature_cols], test_df["SepsisLabel"]

    return X_train, y_train, X_test, y_test


In [5]:
FEATURES_FOR_MODEL = [
    "HR",
    "O2Sat",
    "Temp",
    "SBP",
    "MAP",
    "DBP",
    "Resp",
    "BaseExcess",
    "HCO3",
    "FiO2",
    "pH",
    "PaCO2",
    "SaO2",
    "AST",
    "BUN",
    "Alkalinephos",
    "Calcium",
    "Chloride",
    "Creatinine",
    "Glucose",
    "Lactate",
    "Magnesium",
    "Phosphate",
    "Potassium",
    "WBC",
    "Hct",
    "Hgb",
    "PTT",
    "Fibrinogen",
    "Platelets",
    "Age",
    "Gender",
    "Unit1",
    "Unit2",
    "HospAdmTime",
    "HR_mean_6h",
    "Resp_mean_6h",
    "Lactate_mean_6h",
    "MAP_mean_6h",
    "O2Sat_mean_6h",
    "Temp_mean_6h",
    "SBP_mean_6h",
    "Glucose_mean_6h",
    "HR_std_6h",
    "Resp_std_6h",
    "Lactate_std_6h",
    "MAP_std_6h",
    "O2Sat_std_6h",
    "Temp_std_6h",
    "SBP_std_6h",
    "Glucose_std_6h",
    "HR_diff_1h",
    "HR_diff_3h",
    "O2Sat_diff_1h",
    "O2Sat_diff_3h",
    "Temp_diff_1h",
    "Temp_diff_3h",
    "SBP_diff_1h",
    "SBP_diff_3h",
    "MAP_diff_1h",
    "MAP_diff_3h",
    "DBP_diff_1h",
    "DBP_diff_3h",
    "Resp_diff_1h",
    "Resp_diff_3h",
    "Lactate_diff_1h",
    "Lactate_diff_3h",
    "WBC_diff_1h",
    "WBC_diff_3h",
    "Creatinine_diff_1h",
    "Creatinine_diff_3h",
    "Platelets_diff_1h",
    "Platelets_diff_3h",
    "sofa_cv_score",
    "sofa_resp_score",
    "sofa_renal_score",
    "sofa_coag_score",
    "sofa_total_proxy",
    "shock_index",
    "pulse_pressure",
]

In [6]:
X_train, y_train, X_test, y_test = prepare_target_and_split(data, FEATURES_FOR_MODEL)

ratio = (y_train == 0).sum() / (y_train == 1).sum()


params = {
    "objective": "binary",
    "metric": "average_precision",
    "learning_rate": 0.01,
    "num_leaves": 31,
    "random_state": 42,
}

train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

base_model = lgb.LGBMClassifier(**params)

model = CalibratedClassifierCV(estimator=base_model, method="sigmoid", cv=5)
model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 17675, number of negative: 975740
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.170188 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15671
[LightGBM] [Info] Number of data points in the train set: 993415, number of used features: 80
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.017792 -> initscore=-4.011045
[LightGBM] [Info] Start training from score -4.011045
[LightGBM] [Info] Number of positive: 17675, number of negative: 975740
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.188077 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15662
[LightGBM] [Info] Number of data points in the train set: 993415, number of used features: 80
[LightGBM] [

,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2",LGBMClassifie...ndom_state=42)
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'sigmoid'
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary ` for more details... versionadded:: 0.24",None
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",'auto'
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.01
,n_estimators,100
,subsample_for_bin,200000


In [ ]:
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba > 0.03).astype(int)


roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)

print("\n=== РЕЗУЛЬТАТЫ МОДЕЛИ ===")
print(f"ROC-AUC Score: {roc_auc:.4f}")
print(f"PR-AUC Score  : {pr_auc:.4f}")
print("\nОтчет классификации:")
print(classification_report(y_test, y_pred, digits=3, zero_division=0))


=== РЕЗУЛЬТАТЫ МОДЕЛИ ===
ROC-AUC Score: 0.7724
PR-AUC Score  : 0.0721

Отчет классификации:
              precision    recall  f1-score   support

           0      0.986     0.928     0.956    304619
           1      0.081     0.332     0.130      5822

    accuracy                          0.917    310441
   macro avg      0.534     0.630     0.543    310441
weighted avg      0.969     0.917     0.941    310441



Проанализируем предсказания для одного пациента c сепсисом

In [8]:
sepsis_patients = data[data["SepsisLabel"] == 1]["Patient_ID"].unique()

In [9]:
patient_data = data[
    data["Patient_ID"] == sepsis_patients[random.randint(0, len(sepsis_patients))]
]

In [10]:
patient_preds = model.predict_proba(patient_data[FEATURES_FOR_MODEL])[:, 1]

In [11]:
patient_preds

array([0.01306935, 0.01296414, 0.01871596, 0.01893616, 0.01889558,
       0.01897021, 0.01470515, 0.01458802, 0.01445831, 0.01344602,
       0.01275261, 0.01269809, 0.0126869 , 0.01269809, 0.01229059,
       0.01257802, 0.01267148, 0.01265503, 0.01224575, 0.01225189,
       0.01248761, 0.01253775, 0.01265164, 0.01225189, 0.01210523,
       0.01221782, 0.01262353, 0.01263448, 0.01260186, 0.01219524,
       0.01257242, 0.01269343, 0.01259751, 0.0127046 , 0.01274277,
       0.01283501, 0.01272085, 0.01272085, 0.01272085, 0.01272085])

In [12]:
res_df = pd.DataFrame(
    {
        "probs": patient_preds,
        "true_label": patient_data["SepsisLabel"].values,
        "label@0.2": (patient_preds > 0.2).astype(int),
        "label@0.4": (patient_preds > 0.4).astype(int),
        "label@0.6": (patient_preds > 0.6).astype(int),
    }
)

thresholds = [0.2, 0.4, 0.6]
patient_metrics = []

for t in thresholds:
    col_name = f"label@{t}"

    precision = precision_score(res_df["true_label"], res_df[col_name], zero_division=0)
    recall = recall_score(res_df["true_label"], res_df[col_name], zero_division=0)
    f1 = f1_score(res_df["true_label"], res_df[col_name], zero_division=0)

    patient_metrics.append(
        {
            "Порог": t,
            "Precision (Точность)": round(precision, 3),
            "Recall (Полнота)": round(recall, 3),
            "F1-Score": round(f1, 3),
        }
    )

metrics_df = pd.DataFrame(patient_metrics)

print("=== Почасовые метрики для выбранного пациента ===")
print(metrics_df.to_string(index=False))

=== Почасовые метрики для выбранного пациента ===
 Порог  Precision (Точность)  Recall (Полнота)  F1-Score
   0.2                   0.0               0.0       0.0
   0.4                   0.0               0.0       0.0
   0.6                   0.0               0.0       0.0


In [13]:
res_df

,probs,true_label,label@0.2,label@0.4,label@0.6
0,0.013069,0,0,0,0
1,0.012964,0,0,0,0
2,0.018716,0,0,0,0
3,0.018936,0,0,0,0
4,0.018896,0,0,0,0
5,0.018970,0,0,0,0
6,0.014705,0,0,0,0
7,0.014588,0,0,0,0
8,0.014458,0,0,0,0
9,0.013446,0,0,0,0


In [14]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

rf_model.fit(X_train, y_train)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 150 out of 150 | elapsed:  4.8min finished


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",150
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [15]:
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]
y_pred_rf = (y_pred_proba_rf > 0.15).astype(int)

roc_auc = roc_auc_score(y_test, y_pred_proba_rf)
pr_auc = average_precision_score(y_test, y_pred_proba_rf)

print("\n=== РЕЗУЛЬТАТЫ МОДЕЛИ ===")
print(f"ROC-AUC Score: {roc_auc:.4f}")
print(f"PR-AUC Score  : {pr_auc:.4f}")
print("\nОтчет классификации:")
print(classification_report(y_test, y_pred_rf, digits=3, zero_division=0))

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 150 out of 150 | elapsed:    1.4s finished



=== РЕЗУЛЬТАТЫ МОДЕЛИ ===
ROC-AUC Score: 0.7723
PR-AUC Score  : 0.0634

Отчет классификации:
              precision    recall  f1-score   support

           0      0.997     0.247     0.397    304619
           1      0.024     0.963     0.047      5822

    accuracy                          0.261    310441
   macro avg      0.511     0.605     0.222    310441
weighted avg      0.979     0.261     0.390    310441



In [16]:
patient_preds_rf = rf_model.predict_proba(patient_data[FEATURES_FOR_MODEL])[:, 1]

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 150 out of 150 | elapsed:    0.1s finished


In [17]:
patient_preds_rf

array([0.35101575, 0.18858832, 0.31821708, 0.36637687, 0.31067598,
       0.36999276, 0.23397611, 0.23719338, 0.24029477, 0.21202322,
       0.23220188, 0.21754084, 0.19134594, 0.17087338, 0.15059585,
       0.154169  , 0.1881755 , 0.17942498, 0.16523984, 0.16400321,
       0.14400899, 0.15515946, 0.16137235, 0.1801221 , 0.13695124,
       0.13402453, 0.28393671, 0.33047554, 0.36572711, 0.35602741,
       0.41791669, 0.44606079, 0.41487743, 0.44460723, 0.51470769,
       0.50966223, 0.51026097, 0.52544339, 0.53873042, 0.53182206])